# Component-wise functional gradient boosting

Boosting is gradient descent in function space {cite}`buhlmann2007boosting`. It
minimizes the empirical risk $\mathcal R(f) = \sum_i w_i\,\rho(y_i, f_i)$ over an
additive predictor by taking small steps along the negative gradient,
approximating each step with the best-fitting base learner.

## The algorithm

```{admonition} Algorithm — component-wise gradient boosting
:class: tip

**Input:** loss $\rho$, base learners $g_1,\dots,g_J$, step length $\nu$,
iterations $m_{\text{stop}}$.

1. **Offset.** Initialize $\hat f^{[0]} \equiv \argmin_{c}
   \sum_i w_i\,\rho(y_i, c)$ (a constant).
2. **For** $m = 1, \dots, m_{\text{stop}}$:
   - **Negative gradient.** Compute the pseudo-residuals
     $u_i^{[m]} = -\,\partial \rho(y_i, f)/\partial f$ evaluated at
     $f=\hat f^{[m-1]}_i$.
   - **Fit each base learner** to $u^{[m]}$ by weighted least squares:
     $\hat g_j = \argmin_{g_j} \sum_i w_i\,(u_i^{[m]} - g_j(x_i))^2$.
   - **Select** the best component
     $j^\star = \argmin_j \sum_i w_i\,(u_i^{[m]} - \hat g_j(x_i))^2$.
   - **Update** $\hat f^{[m]} = \hat f^{[m-1]} + \nu\, \hat g_{j^\star}$.
3. **Output** $\hat f^{[m_{\text{stop}}]}$ and the selection path
   $(j^\star_1, \dots, j^\star_{m_{\text{stop}}})$.
```

The loss determines the pseudo-response; the learner fits and selection use weighted least squares:

- **The base-learner fit is always least squares**, regardless of the loss $\rho$.
  The loss enters only through the pseudo-residuals $u$ (the negative-gradient
  step). This is why one engine
  serves every family: changing $\rho$ changes $u$, nothing else.
- **Selection uses the residual sum of squares**, so the component
  that best explains the current gradient is added — a greedy coordinate step in
  function space.



In [1]:
import numpy as np

y = np.array([2.0, 0.5, 1.5, 3.0, 1.0, 2.5])
x1 = np.array([1.0, 0.0, 1.0, 2.0, 0.0, 1.0], dtype=float)
x2 = np.array([0.0, 1.0, 0.0, 1.0, 2.0, 1.0], dtype=float)
n, nu, m_stop = len(y), 0.1, 5

# mboostrs centers each column internally
x1_c = x1 - x1.mean()
x2_c = x2 - x2.mean()

# Offset: constant that minimizes squared error
# risk = sum of squared residuals, the engine's Gaussian risk (no factor 1/2)
f = np.full(n, y.mean())

for m in range(1, m_stop + 1):
    u = y - f                          # negative gradient (Gaussian)
    β1 = x1_c @ u / (x1_c @ x1_c)      # OLS on centered column
    β2 = x2_c @ u / (x2_c @ x2_c)
    sse1 = ((u - β1 * x1_c) ** 2).sum()
    sse2 = ((u - β2 * x2_c) ** 2).sum()
    if sse1 < sse2:
        f += nu * β1 * x1_c
        print(f"m={m}: select x1, β={β1:.3f}, risk={((y-f)**2).sum():.3f}")
    else:
        f += nu * β2 * x2_c
        print(f"m={m}: select x2, β={β2:.3f}, risk={((y-f)**2).sum():.3f}")


m=1: select x1, β=1.147, risk=3.667
m=2: select x1, β=1.032, risk=3.093
m=3: select x1, β=0.929, risk=2.628
m=4: select x1, β=0.836, risk=2.252
m=5: select x1, β=0.753, risk=1.947


## The L2 fit of the gradient as a descent step

Every iteration of boosting solves a surrogate problem: find the base learner
$g_j$ whose fitted values $\hat g_j$ are closest to the negative gradient $u$
in weighted $L^2$ distance. The connection to the original risk follows from
the negative gradient's direction.

For squared-error loss $\rho(y,f) = \tfrac12 (y-f)^2$ the answer is immediate:
the negative gradient $u_i = y_i - f_i$ is exactly the residual, and the
best-fitting base learner decreases the residual sum of squares by construction.
This is "L2Boosting" {cite}`buhlmann2003boosting`, and the per-step risk
decrease can be quantified exactly.

For a general smooth loss, $u^{[m]}$ is the steepest-descent direction of
$\mathcal R$ in real vector space — the direction that (infinitesimally) reduces
the risk the most. Projecting it onto the span of a base learner (by least
squares) gives the closest realizable direction to that ideal. With step length
$\nu$ sufficiently small, the composed step $\nu \hat g_{j^\star}$ still
decreases the risk. {cite:t}`buhlmann2003boosting` showed that with high probability
the risk converges to its minimum as $m \to \infty$. With $\nu = 0.1$ the
in-sample risk usually decreases at every iteration, but a fixed step length
does not guarantee this for every loss; the recorded risk path shows whether it
did.

### Role of the step length $\nu$

$\nu$ controls the granularity of the path. A smaller $\nu$:

- Produces a finer grid of models (more iterations to reach the same risk).
- Changes the sequence of models, not their form: for every $\nu$ the fit is
  a sum of base-learner fits.
- Tends to produce slightly better out-of-sample performance {cite}`buhlmann2007boosting`,
  because each step is a smaller projection, so the path explores more
  combinations before concentrating.

The standard practice is to fix $\nu$ and tune $m_{\text{stop}}$. Halving
$\nu$ roughly doubles the optimal $m_{\text{stop}}$, so the improvement from
smaller $\nu$ is diminishing. The default $\nu = 0.1$ is a robust compromise.

### Correlated predictors and the role of selection

When base learners overlap (e.g. two linear terms in the same predictor, or
strongly correlated predictors), component-wise selection resolves the ambiguity
at each step. The result depends on the order of selection, and the coefficients
are not uniquely identified — the same additive function can be decomposed in
many ways. Consequently, coefficients from `coef()` describe the allocation of signal
along the fitted path. Selection frequencies from
[](stability_selection.ipynb) describe how that allocation changes under
subsampling. Read each coefficient as
"the effect attributed to this variable in the context of the fitted path,"
not as stable structural parameters.

For unambiguous variable selection with correlated predictors, use
{cite}`meinshausen2010stability` stability selection, which resamples the data
and retains only variables selected consistently across subsamples — see
[](stability_selection.ipynb).

## Interpretation as coordinate descent

The algorithm is a forward stagewise (coordinate descent) procedure in the
$(J+1)$-dimensional space spanned by the offset and the $J$ base learners.
Each step selects one coordinate (the learner with the best empirical fit to
the current pseudo-residual) and advances along it by $\nu$. This has two
consequences:

1. **Between-learner competition.** At each step every learner is evaluated, but
   only the best one is updated. This produces automatic variable selection:
   irrelevant or redundant learners are never "chosen," so their coefficients
   remain exactly zero.

2. **Path dependence.** Because the update is sequential and greedy, the
   coefficients depend on the order of selection. A predictor selected early
   absorbs shared signal, leaving less for correlated predictors selected later.
   For fixed data, settings and tie-breaking, the selection rule determines a
   reproducible path. Correlated predictors can receive different shares of the
   fitted signal when the data change.

## Relation to other regularization methods

Component-wise boosting compares with three familiar strategies:

- **Lasso / $\ell_1$-penalized regression** minimizes
  $\|y - X\beta\|_2^2 + \lambda \|\beta\|_1$. The $\ell_1$ penalty drives
  weak coefficients exactly to zero, producing variable selection. Boosting also
  produces sparse models (coefficients are exactly zero for unselected learners),
  but the regularization path is traced by increasing $m_{\text{stop}}$ rather
  than decreasing $\lambda$. Unlike lasso, boosting allows different penalties
  per learner (via `df`), smooth non-linear terms, and non-Gaussian losses — at
  the cost of a greedy per-step approximation rather than a global convex
  optimization.

- **Forward stepwise regression** also adds one variable at a time, but it
  *re-estimates all* coefficients after each addition. Boosting *freezes*
  coefficients already selected and only takes a small step ($\nu$) along
  the new one. This shrinkage is what makes boosting stable in high dimensions:
  a small step does not overcommit to a variable selected early, so later
  iterations can compensate.

- **Random forests** also average many weak learners, but they select splits
  randomly rather than greedily, and they do not produce an additive
  decomposition. Boosting is additive (the final model is a sum of base
  learners), which makes the effect of each predictor inspectable and
  interpretable.

## The boosting hat matrix and degrees of freedom

With small $\nu$ the predictor at iteration $m$ has an approximate linear form

$$
\hat f^{[m]} \approx B_m\, y,
$$

where $B_m$ is a "boosting hat matrix" built from the selected learners' hat
matrices $H_{j^\star_1}, \dots, H_{j^\star_m}$. The degrees of freedom that
`aic()` reports are its trace,

$$
\mathrm{df}(m) = \operatorname{tr}(B_m),
$$

with the offset not counted ($B_0 = 0$), so $\mathrm{df}(0) = 0$ and the df
grows with $m$, quantifying how much the model has fitted the data. Because
$B_m$ depends on the *selection path*, $\mathrm{df}(m)$ is not simply $m$;
it is typically smaller, reflecting the shrinkage from $\nu < 1$.

### Deriving the recursion for $B_m$

The recursion is one line of algebra per step. At iteration $m$ the
pseudo-residual is $u^{[m]} = y - \hat f^{[m-1]}$ (Gaussian case), the
selected learner $j^\star_m$ projects it through its hat matrix
$H_{j^\star_m}$, and the update adds a fraction $\nu$ of that projection:

$$
\hat f^{[m]} = \hat f^{[m-1]} + \nu\, H_{j^\star_m}\, u^{[m]}
            = \hat f^{[m-1]} + \nu\, H_{j^\star_m}\, (y - \hat f^{[m-1]}).
$$

Grouping the $\hat f^{[m-1]}$ terms,

$$
\hat f^{[m]} = \bigl(I - \nu\, H_{j^\star_m}\bigr)\, \hat f^{[m-1]} + \nu\, H_{j^\star_m}\, y.
$$

By induction on $m$ with the base case $\hat f^{[0]} = B_0 y$, $\hat f^{[m]} = B_m y$
where

$$
\boxed{\;B_m = (I - \nu\, H_{j^\star_m})\, B_{m-1} + \nu\, H_{j^\star_m}.\;}
$$

Taking the offset as the mean gives $B_0 = \mathbf{1}\mathbf{1}^\top/n$ (trace 1).
`aic()` counts only the learners: it starts from $B_0 = 0$, so $B_m y$ is the
fitted value minus the offset (centered learners satisfy $H_j \mathbf 1 = 0$).
This is the boosting hat matrix recursion. It depends on the *selection path*
$(j^\star_1, \dots, j^\star_m)$ and on the per-step shrinkage $\nu$, which is
why two paths that select the same learners a different number of times have
different $\mathrm{df}(m)$.

### From $B_m$ to $\mathrm{df}(m)$

Two trace definitions of degrees of freedom are in use, and they answer
different questions. Write the residual sum of squares of the boosting fit as

$$
\mathrm{RSS} = \|y - \hat f^{[m]}\|^2 = \|(I - B_m) y\|^2.
$$

Taking expectations over $y$ with $\mathbb E[y] = \mu$ and
$\operatorname{Cov}(y) = \sigma^2 I$,

$$
\mathbb E[\mathrm{RSS}] = \|(I - B_m)\mu\|^2 + \sigma^2\, \operatorname{tr}\bigl((I - B_m)^\top (I - B_m)\bigr).
$$

The first term is bias squared; the second is $\sigma^2$ times the residual
degrees of freedom. Expanding the trace,

$$
\operatorname{tr}\bigl((I - B_m)^\top (I - B_m)\bigr) = n - 2\operatorname{tr}(B_m) + \operatorname{tr}(B_m B_m^\top).
$$

so the number of degrees of freedom the fit removes from the residuals is
$\operatorname{tr}(2 B_m - B_m B_m^\top)$. The other definition is the
covariance penalty of Mallows' $C_p$ and AIC,
$\sum_i \operatorname{Cov}(\hat f_i, y_i)/\sigma^2 = \operatorname{tr}(B_m)$,
which measures how strongly each fitted value follows its own observation.
For an orthogonal projector ($B_m$ idempotent and symmetric, as in OLS)
$B_m B_m^\top = B_m$ and the two coincide — the familiar OLS degrees of
freedom. Boosting's $B_m$ is a product of damped projections, which is not an
orthogonal projector, so the two traces differ. `aic()` uses
$\operatorname{tr}(B_m)$, the AIC penalty of {cite:t}`buhlmann2007boosting`.
For the six-observation example, after five selections of `x1` with
$B_0 = 0$, $\operatorname{tr}(B_5) = 0.410$ and
$\operatorname{tr}(2B_5 - B_5B_5^\top) = 0.651$; with the offset projector as
$B_0$ both are larger by one.

```{note}
The df of a single penalized base learner, used to calibrate its penalty, is
a separate quantity; [](penalties_df.ipynb) gives its definition.
```



In [2]:
import numpy as np

y = np.array([2.0, 0.5, 1.5, 3.0, 1.0, 2.5])
x1 = np.array([1.0, 0.0, 1.0, 2.0, 0.0, 1.0], dtype=float)
x2 = np.array([0.0, 1.0, 0.0, 1.0, 2.0, 1.0], dtype=float)
n, nu = len(y), 0.1

# Center columns (matching mboostrs behavior)
x1_c = x1 - x1.mean()
x2_c = x2 - x2.mean()
X1 = x1_c.reshape(-1, 1)
X2 = x2_c.reshape(-1, 1)

# Hat matrices of centered scalar OLS
H1 = (X1 / (X1.T @ X1)) @ X1.T          # n×n, rank 1
H2 = (X2 / (X2.T @ X2)) @ X2.T

# Boosting hat matrix recursion
B = np.zeros((n, n))                     # B₀ = 0: the offset is not counted, as in aic()
for j in [1, 1, 1, 1, 1]:               # selection path from above
    H = H1 if j == 1 else H2
    B = (np.eye(n) - nu * H) @ B + nu * H

# Both traces are well below 5 because ν < 1 shrinks each step
print(f"tr(B_5)             = {np.trace(B):.4f}   (the df aic() reports)")
print(f"tr(2B_5 - B_5 B_5') = {np.trace(2 * B - B @ B.T):.4f}   (residual df used)")


tr(B_5)             = 0.4095   (the df aic() reports)
tr(2B_5 - B_5 B_5') = 0.6513   (residual df used)


## Computation in the engine

The Rust engine implements these updates with optimized linear algebra and
buffer reuse (see [](../implementation/engine)):

- Each base learner factorizes its penalized Gram matrix
  $G = X^\top W X + \lambda K$ **once**, at preparation time. The per-step
  "fit" is an $O(p^2)$ back-substitution (or a single division for scalar
  `bols`), not an $O(p^3)$ re-factorization.
- The negative gradient and the running risk are **fused** into a single pass
  over the data, sharing the expensive link evaluations
  and avoiding two separate $O(n)$ scans.
- For Gaussian loss, the engine maintains **cross-Gram** matrices
  $G[j][w] = B_j^\top W B_w$ and per-learner right-hand sides, updating them
  in $O(p^2)$ per iteration instead of re-projecting the residual in $O(n)$.
  The engine stores the inner products between predictors, so that after one coefficient
  changes, the inner products of every predictor with the residual are updated
  from the stored values instead of recomputed from the $n$ rows.
- The **selection** compares weighted residual sums of squares, the quantity
  R `mboost` uses, with the same tie-break rule (the learner listed first
  wins). On the same data and specification both programs therefore select the
  same learner at each iteration; [](../implementation/parity_and_bugs) reports
  the comparisons.

## The two tuning parameters

```{list-table}
:header-rows: 1

* - Parameter
  - Role
  - Typical choice
* - $\nu$ (step length)
  - Shrinkage per step; smaller $\nu$ to finer path, more iterations needed
  - fixed small, e.g. $\nu = 0.1$
* - $m_{\text{stop}}$
  - Model complexity / amount of fitting; the **primary regularizer**
  - chosen by cross-validated risk
```

$\nu$ and $m_{\text{stop}}$ trade off: halving $\nu$ roughly doubles the optimal
$m_{\text{stop}}$ but tends to improve the solution path slightly. The standard
recipe fixes $\nu$ small and tunes $m_{\text{stop}}$
([](model_selection.ipynb)).

## Weak base learners

For the bias/variance trade-off to be controlled by $m_{\text{stop}}$, each base
learner should be a *low-variance, high-bias* fit — a single linear term, or a
spline penalized to a small `df` (typically $\mathrm{df}=4$). Penalizing the
learners (rather than using rich, unpenalized fits) keeps the per-step increment
small and the selection meaningful; it also equalises the competition between
learners of different flexibility, since selection compares residual sums of
squares. [](base_learners.ipynb) and [](penalties_df.ipynb) describe the
learners and their penalties.

Choosing `df` too large collapses the contest: a rich spline with `df=20`
produces predictions that are closer to the pseudo-residual than any linear
term, so it is selected every iteration and the model degrades to running one
large step per component. Choosing `df` too small makes the effect negligible
— the learner never wins selection and effectively drops out. The default
`df=4` (2 degrees of freedom above the null space for a second-order P-spline,
which already includes a linear function for free) is a robust starting point;
see [](penalties_df.ipynb) for the mechanics and [](base_learners.ipynb) for
practical guidance.

For a one-step penalized B-spline fit, `df=4` means that the fitted curve has
four effective degrees of freedom. In boosting, the base-learner `df` instead
sets the flexibility of each update. The `df` on a base learner sizes *one step*;
the model is a sum of $m_{\text{stop}}$ such steps, and its complexity is
governed by $m_{\text{stop}}$, through the boosting hat matrix $B_m$ above
whose trace grows with $m$. Iterating far enough therefore removes the
penalty's influence: keeping $\nu$ fixed and letting $m \to \infty$ drives
the fit toward the unpenalized least-squares projection onto the learner's
basis, whatever `df` was chosen. The
same `df=4` that would regularize a one-shot fit is eventually overridden by
a long run. Early stopping by cross-validated risk is what controls
that — see [](model_selection.ipynb).



## Gradient and offset checks

Two properties of each loss family can be checked numerically. The checks
evaluate the offset, negative gradient and risk of a family at a supplied
point $(y, f, w)$, using the same routines as the fitting loop:

1. **The negative gradient is (proportional to) minus the derivative of the
   risk.** Central-difference the risk in each coordinate of `f` and compare.
2. **The offset minimizes the risk over a constant predictor** — re-derived
   here by direct scalar minimization rather than trusted from the formula.

```{admonition} Why "proportional to", not "equal to"
:class: note
Some families report a `risk()` scaled differently from the loss whose
derivative is `negative_gradient()`. This does not affect fitting (only the
*ratio* between successive risk values and the gradient direction matter),
but it means a literal `u == -d(risk)/df` check is too strict. The gradient-ratio table
checks that the ratio is **constant across observations**, which
catches sign errors, wrong-argument bugs, and mismatched formulas just as
well, and separately reports what that constant is.
```


In [3]:
import numpy as np
import polars as pl
import mboostrs as mb
import mboostrs._rust as _rust
from mboostrs.families import _family_rust_name, _family_params

rng = np.random.default_rng(0)
n = 8
w = np.ones(n)


def gradient_ratio(name, params, y, f, w, h=1e-6):
    """grad[i] / (-d(risk)/df_i), central difference; entries with a
    near-zero derivative are dropped (0/0 is uninformative, not a failure)."""
    _, grad, _ = _rust.family_diagnostics(name, params, y, f, w)
    raw = np.zeros_like(f)
    for i in range(len(f)):
        fp, fm = f.copy(), f.copy()
        fp[i] += h
        fm[i] -= h
        _, _, r_plus = _rust.family_diagnostics(name, params, y, fp, w)
        _, _, r_minus = _rust.family_diagnostics(name, params, y, fm, w)
        raw[i] = -(r_plus - r_minus) / (2 * h)
    mask = np.abs(raw) > 1e-8
    return grad[mask] / raw[mask]


# Standard families: one scalar y, f, w per observation.
standard_cases = {
    "Gaussian": rng.normal(size=n),
    "GaussReg": rng.normal(size=n),
    "Laplace": rng.normal(size=n),
    "Quantile": rng.normal(size=n),
    "Expectile": rng.normal(size=n),
    "Poisson": rng.integers(1, 10, size=n).astype(float),
    "GammaReg": rng.uniform(0.5, 5, size=n),
    "NBinomial": rng.integers(1, 10, size=n).astype(float),
    "Binomial": rng.choice([-1.0, 1.0], size=n),
    "AdaExp": rng.choice([-1.0, 1.0], size=n),
    "HingeLoss": rng.choice([-1.0, 1.0], size=n),
}

rows = []
for cname, y in standard_cases.items():
    obj = getattr(mb, cname)()
    name, params = _family_rust_name(obj), _family_params(obj)
    f = rng.normal(size=n) * 0.3
    ratio = gradient_ratio(name, params, y, f, w)
    rows.append({"family": cname, "ratio (mean)": ratio.mean(), "ratio (std)": ratio.std()})

# Survival families: y = concat(time, status), same scalar f, w.
time = rng.uniform(0.5, 5, size=n)
status = rng.choice([0.0, 1.0], size=n)
y_surv = np.concatenate([time, status])
for cname in ["CoxPH", "Weibull", "Loglog", "Lognormal"]:
    obj = getattr(mb, cname)()
    name = _family_rust_name(obj)
    f = rng.normal(size=n) * 0.3
    ratio = gradient_ratio(name, {}, y_surv, f, w)
    rows.append({"family": cname, "ratio (mean)": ratio.mean(), "ratio (std)": ratio.std()})

results = pl.DataFrame(rows).with_columns(
    (pl.col("ratio (std)") < 1e-4).alias("consistent")
)
results


family,ratio (mean),ratio (std),consistent
str,f64,f64,bool
"""Gaussian""",0.5,1.3793e-10,true
"""GaussReg""",0.5,5.4447e-10,true
"""Laplace""",1.0,1.9028e-10,true
"""Quantile""",1.0,2.1499e-10,true
"""Expectile""",1.0,1.1715e-10,true
…,…,…,…
"""HingeLoss""",1.0,3.8459e-10,true
"""CoxPH""",1.0,1.3559e-9,true
"""Weibull""",1.0,1.0590e-9,true


Every family passes: the gradient is exactly proportional to minus the risk
derivative, observation by observation. Three ratios are worth reading:

- Most families: ratio $= 1$ — `negative_gradient` is exactly
  $-\partial\rho/\partial f$.
- **Gaussian / GaussReg: ratio $= 0.5$.** The engine reports
  $\text{risk} = \sum_i w_i (y_i - f_i)^2$ (no $\tfrac12$), matching R
  `mboost`'s convention for readable SSE-scale risk values, while the
  gradient $u = y - f$ is the derivative of the *textbook* $\tfrac12(y-f)^2$
  loss. Fitting is unaffected — boosting only uses the gradient's
  *direction*, and risk is only compared to itself along one path — but a
  literal $u = -\partial\,\text{risk}/\partial f$ check would fail by a
  factor of 2.
- **Binomial: ratio $= 1/\ln 2 \approx 1.443$.** `risk()` is the natural-log
  loss $\ln(1 + e^{-2yf})$. `negative_gradient`, like R `mboost`'s, is the
  derivative of the base-2 loss $\log_2(1 + e^{-2yf})$ of
  {cite:t}`buhlmann2007boosting`, so it carries the factor $1/\ln 2$.

Two families are left out of the gradient-ratio table:

```{admonition} Huber: risk() and negative_gradient() use different deltas
:class: warning
Huber's robustifying threshold $\delta$ defaults to an adaptive,
**state-dependent** value: `negative_gradient` recomputes it every call as
the median $|y - f_{\text{prev}}|$ from the *previous* iteration's fit (R
`mboost`'s `Huber()` closure keeps exactly this one-iteration lag), while
`risk()` uses whatever fixed $\delta$ the family was constructed with. A
finite-difference check of `risk()` therefore differentiates a *different*
$\delta$ than the one `negative_gradient` used, and the two disagree by
more than numerical noise. The lag matches R. Checking Huber's gradient therefore requires holding
$\delta$ fixed on both sides rather than using the generic finite-difference check,
as in the R comparison
([](../implementation/parity_and_bugs)).
```

```{admonition} Multinomial: risk() omits the reference category's likelihood term
:class: warning
For an observation whose true class *is* the dropped reference category, every
`y` entry in its row is 0, so `risk()`'s `if y_ik > 0` guard never fires and
that observation contributes exactly **0** to the reported risk — regardless
of `f`. `negative_gradient` has no such gap (it returns $y_{ik} - p_{ik}$,
correctly nonzero for reference-category rows), so the two disagree there by
construction, not by a bounded ratio. Checked directly: `risk()` is flat at
`0.0` for a reference-category row across widely different `f`, while
`negative_gradient` still varies with `f` as expected. R `mboost` reports the
same risk values, so the omission follows R. Fitting uses `negative_gradient`, which includes
reference-category rows. The reported risk omits their likelihood terms.
Those omitted terms depend on the fitted probabilities, so the reported risk
is not the full deviance and can rank iteration counts differently from it.
```

### Offset: the risk-minimizing constant

The claim is $\hat f^{[0]} = \arg\min_c \sum_i w_i \rho(y_i, c)$. Some
families' risk is only piecewise-linear in $c$ (Laplace's weighted median),
so their minimizer isn't a single point — comparing *locations* would fail
for reasons that have nothing to do with correctness. Comparing **risk
values** is robust to that: if `offset` minimizes the risk, no other
point can do better.


In [4]:
from scipy.optimize import minimize_scalar

offset_cases = {
    "Gaussian": (rng.normal(size=n), {}),
    "Poisson": (rng.integers(1, 10, size=n).astype(float), {}),
    "GammaReg": (rng.uniform(0.5, 5, size=n), {}),
    "Binomial": (rng.choice([-1.0, 1.0], size=n), {}),
    "NBinomial": (rng.integers(1, 10, size=n).astype(float), {}),
    "Laplace": (rng.normal(size=n), {}),
}

rows = []
for cname, (y, extra_params) in offset_cases.items():
    obj = getattr(mb, cname)()
    name = _family_rust_name(obj)
    params = {**_family_params(obj), **extra_params}
    offset, _, _ = _rust.family_diagnostics(name, params, y, np.zeros(n), w)

    def risk_at(c, name=name, params=params, y=y):
        _, _, risk = _rust.family_diagnostics(name, params, y, np.full(n, c), w)
        return risk

    found = minimize_scalar(risk_at, bracket=(offset - 2, offset, offset + 2))
    rows.append({
        "family": cname, "offset": offset,
        "risk(offset)": risk_at(offset), "min risk found": found.fun,
        "offset is optimal": risk_at(offset) <= found.fun + 1e-6,
    })

pl.DataFrame(rows)


family,offset,risk(offset),min risk found,offset is optimal
str,f64,f64,f64,f64
"""Gaussian""",-0.369037,5.409918,5.409918,1.0
"""Poisson""",1.658228,19.572747,19.572747,1.0
"""GammaReg""",1.05294,9.860961,9.860961,1.0
"""Binomial""",0.0,5.545177,5.545177,1.0
"""NBinomial""",1.446919,20.450334,20.450334,1.0
"""Laplace""",-0.886798,6.53868,6.53868,1.0


```{admonition} HingeLoss's offset does not minimize its own risk
:class: warning
HingeLoss's risk, $\sum_i w_i \cdot \text{cost}(y_i) \cdot \max(0, 1-y_i c)$,
is piecewise-*linear* in a constant $c$, with slope $W_- - W_+$ on $[-1, 1]$
(the cost-weighted class masses) and strictly worse outside it. Its
minimizer is therefore always a boundary point, $c = \pm 1$; no interior
offset can minimize it. `HingeLoss.offset()` instead returns
$\tfrac12\ln(p/(1-p))$, the AdaBoost/exponential-loss log-odds formula. A
direct scan of the risk confirms the two disagree. With 5 observations of
class $+1$, 3 of class $-1$, unit weights and $\alpha_{\text{cost}} = 0.5$
(cost $1 - \alpha_{\text{cost}}$ for class $+1$ and $\alpha_{\text{cost}}$
for class $-1$), the formula gives the offset $\tfrac12\ln(5/3) = 0.255$ with
risk $0.5\,[5(1 - 0.255) + 3(1 + 0.255)] = 3.74$, against the minimum
$0.5 \cdot 3 \cdot 2 = 3.00$ at $c = 1$.

R's `mboost` has an internal (non-exported) `HingeLoss()` family —
`mboost:::HingeLoss`, used by `Binomial(type = "adaboost")` — and its offset
is the identical `1/2 * log(p/(1 - p))`. `mboostrs` reproduces R's value
exactly.
```


## Summary

- Component-wise functional gradient boosting fits a separate base learner to
  the negative gradient of the loss at each iteration and adds a fraction
  $\nu$ of the best fit to the predictor. Only the winning learner's parameters
  change — all others stay frozen.
- The boosting hat matrix $B_m$ satisfies the recursion
  $B_m = (I - \nu H_{j^\star_m}) B_{m-1} + \nu H_{j^\star_m}$, and its
  trace $\operatorname{tr}(B_m)$ is the degrees of freedom `aic()` uses (the
  covariance penalty of Mallows' $C_p$); $\operatorname{tr}(2B_m - B_m B_m^\top)$
  counts the residual degrees of freedom the fit uses.
- The algorithm has two tuning parameters: $m_{\text{stop}}$ (the primary
  regularizer, controlling model complexity) and $\nu$ (step-length shrinkage,
  a secondary regularizer that interacts with selection).

## See also

- [](worked_example.ipynb) — the algorithm above traced on real numbers.
- [](from_glm_to_boosting.ipynb) — the historical path from GLMs to this
  algorithm, motivating each design choice.
- [](gamlss.ipynb) — the multi-parameter (cyclic / non-cyclic) extension.
- [](model_selection.ipynb) — choosing $m_{\text{stop}}$ by cross-validation or
  AIC.
- [](../implementation/engine) — the Rust implementation of this loop.